## 1_download_VIDA_datasets_Kenya

The notebook downloads VIDA country-level building data from ```data.source.coop```. In the first cell, configuration parameters for connection to IBM COS are defined (for this notebook, no utilities are downloaded from the cloud, only used to upload final data to COS bucket). The next cells are structured as follows: import the necessary libraries, define the country of interest and corresponding ISO, inialize IBM COS s3 client for connection, define a helper function that
- creates a directory for storing downloaded data (if it does not exist)
- builds a URL for downloading the data based on the country ISO defined
- downloads the GeoParquet file in streaming mode with a progress bar
- uploads the downloaded file to the configured COS bucket.

The function is called in the last cell.  


In [ ]:
# Read notebook configuration
import getpass
import json

# config_str = getpass.getpass('Enter your prepared config: ')
config_str = '''

{
"COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
"COUNTRY_NAME": "Kenya",
"OUTPUT_BUCKET": "",
"UTILS_BUCKET": "",

"COS_AUTH_ENDPOINT_URL": "",
"COS_APIKEY": ""
}

'''

config = json.loads(config_str)

In [9]:
# Import necessary libraries
import requests
import os
from botocore.client import Config
import ibm_boto3
from tqdm import tqdm
import io

In [10]:
# countries ISO mapper - Add new countries if needed
country_mapper = {
    'Kenya': 'KEN'
}

In [11]:
# # init S3 client 
cos_client = ibm_boto3.client(service_name='s3',
                                  ibm_api_key_id=config["COS_APIKEY"],
                                  ibm_auth_endpoint="https://iam.cloud.ibm.com/oidc/token",
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])


response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

utils_to_download = [] # no utilities are downloaded atm

try:
    for obj in response['Contents']:
        name = obj['Key']
        if name in utils_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)
    
except Exception as e:
    print('Error occured: ', e)

In [12]:
def download_country_parquet_by_country(country:str, directory:str, target_bucket=None, remove_after_upload:bool=False) -> str:
    '''
        This function is aimed for downloading VIDA building footprint data for an entire country from data.source.coop
        Input positional arguments:
            1. country -> country name, set to Kenya,
            2. directory -> target directory where desired parquet will be saved
            3. target_bucket -> (optional) if defined the downloaded parquet will be uploaded to the bucket assigned to this argumemt
        
    '''
    
    # check desired directory existence
    if os.path.exists(directory):
        print(f'\033[92mDirectory: "{directory}" exists')
        
    else:
        print(f'\033[93mTarget directory not exists, creating...')
        
        try:
            os.makedirs(directory)
            print(f'\033[92mDirectory "{directory}" successfully created')
            
        except Exception as e:
            print(f"\033[91mError occurred while creating directory {directory} \n Error: {str(e)}")
    
    # assembly final url
    country_iso = country_mapper[country]
    url = f'https://data.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country/country_iso={country_iso}/{country_iso}.parquet'

    filename = f"{country_iso}.parquet"
    file_path = os.path.join(directory, filename)
    
    # get file size
    
    try:
        response = requests.head(url, allow_redirects=True)
        size = response.headers.get('content-length', -1)
        # size in megabytes
        print('FILE SIZE: {:.2f} MB'.format(int(size) / float(1 << 20)))
        
    except Exception  as e:
        print(f'Headers retrieval error {e}')
        
        
    try:
        # download file
        print(f'Downloading VIDA data for {country} from {url} ...')
        response = requests.get(url, stream=True, timeout=120)
        
        if response.status_code == 200:
            chunk_size = 8192
            total = int(response.headers.get('content-length', 0))
            
            with open(file_path, "wb") as file, tqdm(total=total, unit="B", unit_scale=True, desc=f"Progress ({filename})", ncols=80) as pbar:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        file.write(chunk)
                        pbar.update(len(chunk))
            print(f'File "{filename}" downloaded successfully!')
        else:
            raise RuntimeError(f'Download failed — HTTP {response.status_code}')
    except Exception as e:
        print(f'\033[91mDownload error: {e}\033[0m')
        raise


     # ---- optional upload to COS ----
        
    if isinstance(target_bucket, str):
        try:
            cos_client.upload_file(
                Filename=file_path,
                Bucket=target_bucket,
                Key=filename,
                ExtraArgs={'ContentDisposition': 'attachment'}
            )
            
            if remove_after_upload:
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(f"\033[91mFailed to remove {file_path}")
            
            print(f'\033[92mFile {filename} successfully uploaded to the COS {target_bucket} bucket')
        except Exception as e:
            print(f"\033[91mFailed upload file to the bucket {target_bucket}. Error: {e}")

    return file_path         
   

In [13]:
country_name = config["COUNTRY_NAME"]
output_dir = "parquets"

country_parquet = download_country_parquet_by_country(
    country_name,
    output_dir,
    target_bucket=config["OUTPUT_BUCKET"],
    remove_after_upload=False
)


Directory: "parquets" exists
FILE SIZE: 3382.70 MB


Progress (KEN.parquet): 100%|██████████████| 3.55G/3.55G [03:07<00:00, 18.9MB/s]


File "KEN.parquet" downloaded successfully!
File KEN.parquet successfully uploaded to the COS onboarding-bucket-s bucket
